In [40]:
config = {
    # header:
    "version": "0.1",
    "task": "potential",
    # vairables:
    "var": {
        "cutoff": 5.0,
        # theoretically we can init class and use its instance as a variable
    },
    # defination:
    "data": {
        "source": {
            "class_path": "mpot.dataset.rMD17",
            "type": "builtin",
            "kwargs": {
                "molecule": "aspirin",
                "save_dir": "./rmd17",
                "total": 1000,
                "device": "cuda",
            },
            "processes": [
                {"name": "NeighborList", "kwargs": {"cutoff": 5.0}},
            ],
        },
        # "loader": {
        #     "split": {
        #         "train": 0.95,
        #         "eval": 0.05,
        #     },
        #     "loaders": [
        #         {
        #             "name": "train",
        #             "batch_size": 10,
        #             "shuffle": True,
        #         },
        #         {
        #             "name": "eval",
        #             "batch_size": 10,
        #             "shuffle": False,
        #         },
        #     ],
        # },
    },
    "model": {
        "container": "PotentialSeq",
        "@modules": [
            {
                "class": "mpot.potential.nnp.PiNet",
                "kwargs": {
                    "depth": 5,
                    "@basis_fn": {
                        "class": "mpot.potential.nnp.radial.GaussianRBF",
                        "kwargs": {
                            "n_basis": 10,
                            "cutoff": "$cutoff",
                        },
                    },
                    "@cutoff_fn": {
                        "class": "mpot.potential.nnp.radial.CosineCutoff",
                        "kwargs": {
                            "cutoff": "$cutoff",
                        },
                    },
                    "pi_nodes": [64, 64],
                    "ii_nodes": [64, 64, 64, 64],
                    "pp_nodes": [64, 64, 64, 64],
                    "@activation": {
                        "class": "torch.nn.Tanh",
                    },
                    "rank": 1,
                },
            },
            {
                "class": "mpot.potential.nnp.readout.Atomwise",
                "args": [[64, 1]],
                "kwargs": {
                    "from_key": ("pinet", "p1"),
                    "to_key": ("predicts", "energy"),
                },
            },
            {
                "class": "mpot.potential.nnp.readout.DPairPot",
                "kwargs": {
                    "fx_key": ("predicts", "energy"),
                    "dx_key": ("pairs", "dist"),
                    "to_key": ("predicts", "forces"),
                    "create_graph": True,
                },
            },
        ],
    },
    "trainer": {
        "@loss_fn": {
            "class": "mpot.loss.MultiTargetLoss",
            "kwargs": {
                "@kernel": "torch.nn.MSELoss",
                "keys": [("forces", "forces", 1.0)]
            }
        },
        "@optimizer": {
            "class": "torch.optim.Adam",
            "kwargs": {
                "lr": 1e-4,
            },
        },
        "device": "cuda",
    }
}

In [ ]:
from jsonargparse import ArgumentParser, lazy_instance
from pathlib import Path
import json

In [ ]:
parser = ArgumentParser()
parser.add_argument("--config", action="config")
parser.add_argument("task", type=str, help="Task to perform")
parser.add_argument("data.source.class_path", type=str, help="Class path for the source dataset", action=lazy_instance)
parser.add_argument("data.source.type", type=str, choices=["builtin", "external"], help="Source type")
parser.add_argument("data.source.kwargs", type=dict, help="Additional keyword arguments for the source dataset")
parser.add_argument("data.source.processes", type=list, help="List of processing steps")
# parser.add_argument("data.source.dict_kwargs")


ActionTypeHint(option_strings=[], dest='data.source.processes', nargs=None, const=None, default=None, type=None, choices=None, required=False, help='List of processing steps', metavar=None)

In [46]:
cfg = parser.parse_args(["--config", json.dumps(config)])

In [49]:
cfg.data.source.class_path

'mpot.dataset.rMD17'